In [2]:
import kagglehub
import pandas as pd
import numpy as np
import ast

In [3]:
path = kagglehub.dataset_download(
    "rounakbanik/the-movies-dataset"
)
metaDataDF = pd.read_csv(f"{path}/movies_metadata.csv")
ratingsDF = pd.read_csv(f"{path}/ratings_small.csv")


C:\Users\Johnny\AppData\Local\Temp\ipykernel_25524\3779643782.py:4: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  metaDataDF = pd.read_csv(f"{path}/movies_metadata.csv")


# Dataset - The Movies Dataset by Rounak Banik
report by John Lopes & Taha Riyaan

## Description

"About Dataset
Context
These files contain metadata for all 45,000 movies listed in the Full MovieLens Dataset. The dataset consists of movies released on or before July 2017. Data points include cast, crew, plot keywords, budget, revenue, posters, release dates, languages, production companies, countries, TMDB vote counts and vote averages.

This dataset also has files containing 26 million ratings from 270,000 users for all 45,000 movies. Ratings are on a scale of 1-5 and have been obtained from the official GroupLens website.

Content
This dataset consists of the following files:

movies_metadata.csv: The main Movies Metadata file. Contains information on 45,000 movies featured in the Full MovieLens dataset. Features include posters, backdrops, budget, revenue, release dates, languages, production countries and companies.

keywords.csv: Contains the movie plot keywords for our MovieLens movies. Available in the form of a stringified JSON Object.

credits.csv: Consists of Cast and Crew Information for all our movies. Available in the form of a stringified JSON Object.

links.csv: The file that contains the TMDB and IMDB IDs of all the movies featured in the Full MovieLens dataset.

links_small.csv: Contains the TMDB and IMDB IDs of a small subset of 9,000 movies of the Full Dataset.

ratings_small.csv: The subset of 100,000 ratings from 700 users on 9,000 movies.

The Full MovieLens Dataset consisting of 26 million ratings and 750,000 tag applications from 270,000 users on all the 45,000 movies in this dataset can be accessed here

Acknowledgements
This dataset is an ensemble of data collected from TMDB and GroupLens.
The Movie Details, Credits and Keywords have been collected from the TMDB Open API. This product uses the TMDb API but is not endorsed or certified by TMDb. Their API also provides access to data on many additional movies, actors and actresses, crew members, and TV shows. You can try it for yourself here.

The Movie Links and Ratings have been obtained from the Official GroupLens website. The files are a part of the dataset available here



Inspiration
This dataset was assembled as part of my second Capstone Project for Springboard's Data Science Career Track. I wanted to perform an extensive EDA on Movie Data to narrate the history and the story of Cinema and use this metadata in combination with MovieLens ratings to build various types of Recommender Systems.

Both my notebooks are available as kernels with this dataset: The Story of Film and Movie Recommender Systems

Some of the things you can do with this dataset:
Predicting movie revenue and/or movie success based on a certain metric. What movies tend to get higher vote counts and vote averages on TMDB? Building Content Based and Collaborative Filtering Based Recommendation Engines."

We decided to go with this dataset because we enjoying watching movies

## Code

### Cleaning

### Study 1

subset = genres, revenue, runtime, production_companies, title

Jaccard

In [3]:
def extractGenreNames(genres):
    if pd.isna(genres):
        return set()
    if isinstance(genres, str):
        try:
            genres = ast.literal_eval(genres)
        except (ValueError, SyntaxError):
            return set()
    if isinstance(genres, list):
        return {item['name'] for item in genres if isinstance(item, dict) and 'name' in item}
    return set()

def jaccardSimilarity(a, b):
    if not a and not b:
        return 1.0
    union = a | b
    return len(a & b) / len(union) if union else 0.0

movie = "Minions"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieGenres = extractGenreNames(movieRow.iloc[0]['genres']) if not movieRow.empty else set()
print(movie + " genres: " + str(movieGenres))

jaccardScores = []
for title in metaDataDF['title']:
    movieRow = metaDataDF[metaDataDF['title'] == title]
    if movieRow.empty:
        movieRow = metaDataDF[metaDataDF['original_title'] == title]
    genres = extractGenreNames(movieRow.iloc[0]['genres']) if not movieRow.empty else set()
    jaccardScores.append({
        'title': title,
        'genres': genres,
        'jaccard_similarity': jaccardSimilarity(movieGenres, genres),
    })

jaccardDF = pd.DataFrame(jaccardScores).sort_values('jaccard_similarity', ascending=False)
jaccardDF = jaccardDF[jaccardDF['title'] != movie]
jaccardDF.head(10)

Minions genres: {'Comedy', 'Family', 'Adventure', 'Animation'}


,title,genres,jaccard_similarity
8610,Asterix the Gaul,"{Comedy, Family, Adventure, Animation}",1.0
8626,Asterix and Cleopatra,"{Comedy, Family, Adventure, Animation}",1.0
43156,Smurfs: The Lost Village,"{Adventure, Family, Comedy, Animation}",1.0
608,The Aristocats,"{Family, Comedy, Adventure, Animation}",1.0
41586,Tri bogatyrya i Shamakhanskaya tsaritsa,"{Family, Comedy, Adventure, Animation}",1.0
40608,Storks,"{Adventure, Family, Comedy, Animation}",1.0
39750,VeggieTales: Tomato Sawyer & Huckleberry Larry...,"{Family, Comedy, Adventure, Animation}",1.0
39734,VeggieTales: Dave and the Giant Pickle,"{Adventure, Family, Comedy, Animation}",1.0
43294,Cars 3,"{Family, Comedy, Adventure, Animation}",1.0
39334,Ice Age: Collision Course,"{Adventure, Family, Comedy, Animation}",1.0


In [4]:
def extractCompanyNames(companies):
    if pd.isna(companies):
        return set()
    if isinstance(companies, str):
        try:
            companies = ast.literal_eval(companies)
        except (ValueError, SyntaxError):
            return set()
    if isinstance(companies, list):
        return {item['name'] for item in companies if isinstance(item, dict) and 'name' in item}
    return set()

movie = "Pocahontas"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieCompanies = extractCompanyNames(movieRow.iloc[0]['production_companies']) if not movieRow.empty else set()
print(movie + " production companies: " + str(movieCompanies))

jaccardScores = []
for title in metaDataDF['title']:
    movieRow = metaDataDF[metaDataDF['title'] == title]
    if movieRow.empty:
        movieRow = metaDataDF[metaDataDF['original_title'] == title]
    companies = extractCompanyNames(movieRow.iloc[0]['production_companies']) if not movieRow.empty else set()
    jaccardScores.append({
        'title': title,
        'companies': companies,
        'jaccard_similarity': jaccardSimilarity(movieCompanies, companies),
    })

jaccardDF = pd.DataFrame(jaccardScores).sort_values('jaccard_similarity', ascending=False)
jaccardDF = jaccardDF[jaccardDF['title'] != movie]
jaccardDF.head(10)

Pocahontas production companies: {'Walt Disney Pictures', 'Walt Disney Feature Animation'}


,title,companies,jaccard_similarity
1798,Mulan,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
5744,Treasure Planet,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
10520,Chicken Little,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
359,The Lion King,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
4238,Atlantis: The Lost Empire,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
5310,Lilo & Stitch,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
3493,Dinosaur,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
3891,The Emperor's New Groove,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
1980,The Rescuers Down Under,"{Walt Disney Pictures, Silver Screen Partners ...",0.666667
1520,Air Bud,{Walt Disney Pictures},0.500000


Manhattan

In [5]:
movie = "Mortal Kombat"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieRevenue = movieRow.iloc[0]['revenue'] if not movieRow.empty else 0.0
print(movie + " revenue: " + str(movieRevenue))

manhattanDistances = []
for i, row in metaDataDF.iterrows():
    rev = row['revenue']
    if pd.isna(rev) or pd.isna(movieRevenue):
        dist = float('inf')
    else:
        dist = abs(rev - movieRevenue)
    manhattanDistances.append({
        'title': row['title'],
        'revenue': rev,
        'manhattan_distance': dist,
    })

manhattanDF = pd.DataFrame(manhattanDistances)
manhattanDF = manhattanDF[manhattanDF['title'] != movie]
manhattanDF = manhattanDF.sort_values('manhattan_distance')
manhattanDF.head(10)

Mortal Kombat revenue: 122195920.0


,title,revenue,manhattan_distance
1885,Poltergeist,122200000.0,4080.0
14494,Invictus,122233971.0,38051.0
21604,Prisoners,122126687.0,69233.0
7011,Peter Pan,121975011.0,220909.0
1631,MouseHunt,122417389.0,221469.0
489,Executive Decision,121969216.0,226704.0
14234,Surrogates,122444772.0,248852.0
10762,Nanny McPhee,122489822.0,293902.0
31322,Magic Mike XXL,122513057.0,317137.0
5255,Spirit: Stallion of the Cimarron,122563539.0,367619.0


Euclidean

In [6]:
movie = "Avatar"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieRuntime = movieRow.iloc[0]['runtime'] if not movieRow.empty else 0.0
print(movie + " runtime: " + str(movieRuntime))

euclideanDistances = []
for i, row in metaDataDF.iterrows():
    rt = row['runtime']
    if pd.isna(rt) or pd.isna(movieRuntime):
        dist = float('inf')
    else:
        dist = np.sqrt((rt - movieRuntime) ** 2)
    euclideanDistances.append({
        'title': row['title'],
        'runtime': rt,
        'euclidean_distance': dist,
    })

euclideanDF = pd.DataFrame(euclideanDistances)
euclideanDF = euclideanDF[euclideanDF['title'] != movie]
euclideanDF = euclideanDF.sort_values('euclidean_distance')
euclideanDF.head(10)

Avatar runtime: 162.0


,title,runtime,euclidean_distance
38801,Toni Erdmann,162.0,0.0
14992,Bunty Aur Babli,162.0,0.0
18451,The Disappearance of Haruhi Suzumiya,162.0,0.0
36560,Garv: Pride and Honour,162.0,0.0
33610,Times of Joy and Sorrow,162.0,0.0
29622,Sarfarosh,162.0,0.0
33434,A Tale of Two Cities,162.0,0.0
12520,Marketa Lazarová,162.0,0.0
7934,How the West Was Won,162.0,0.0
36571,Rakht,162.0,0.0


Edit Distance

In [ ]:
def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previousRow = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        currentRow = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previousRow[j + 1] + 1
            deletions = currentRow[j] + 1
            substitutions = previousRow[j] + (c1 != c2)
            currentRow.append(min(insertions, deletions, substitutions))
        previousRow = currentRow
    return previousRow[-1]

movie = "Pacific Rum"
editDistances = []
for title in metaDataDF['title']:
    dist = levenshtein(movie, str(title))
    editDistances.append({
        'title': title,
        'edit_distance': dist,
    })

editDF = pd.DataFrame(editDistances).sort_values('edit_distance')
editDF.head(10)

,title,edit_distance
21123,Pacific Rim,1
5132,Panic Room,5
26192,Sacrifice,6
38348,Sacrifice,6
18385,Sacrifice,6
40995,Pacific Banana,6
30770,Traffic Jam,6
9239,Mimic 2,7
11792,Maniac Cop,7
35942,Panic,7


### Study 2

### Study 3

### Study 4

In [ ]:
userItemMatrix = ratingsDF.pivot(index='userId', columns='movieId', values='rating').fillna(0)
userItemArray = userItemMatrix.to_numpy()
userItemArrayValidation = userItemArray.copy()

In [ ]:
np.random.seed(17)

numToRemove = int(len(userItemArray) * 0.1)
removeIndices = np.random.choice(len(userItemArray), numToRemove, replace=False)
for idx in removeIndices:
    for i in range(len(userItemArrayValidation[idx])):
        userItemArrayValidation[idx][i] = 0

In [ ]:
def matrix_factorization(R, P, Q, K, steps=500, alpha=0.0004, beta=0.02):
    Q = Q.T
    error=[]
    for step in range(steps):
        for i in range(len(R)):
            for j in range(len(R[i])):
                if R[i][j] > 0:
                    eij = R[i][j] - np.dot(P[i,:],Q[:,j])
                    for k in range(K):
                        P[i][k] = P[i][k] + alpha * (2 * eij * Q[k][j] - beta * P[i][k])
                        Q[k][j] = Q[k][j] + alpha * (2 * eij * P[i][k] - beta * Q[k][j])
        eR = np.dot(P,Q)
        e = 0
        for i in range(len(R)):
            for j in range(len(R[i])):
                if R[i][j] > 0:
                    e = e + pow(R[i][j] - np.dot(P[i,:],Q[:,j]), 2)
                    for k in range(K):
                        e = e + (beta/2) * (pow(P[i][k],2) + pow(Q[k][j],2))
        error.append(e)
        if e < 0.001:
            break
    # for i in range(len(error)):
    #     print(error[i])    
    return P, Q.T

R = np.array(userItemArrayValidation)
N = len(R)
M = len(R[0])
K = 10
P = np.random.rand(N,K)
Q = np.random.rand(M,K)
nP, nQ = matrix_factorization(R, P, Q, K)
nR = np.dot(nP, nQ.T)

171537.57721497118
141995.14121080638
127861.69600039083
119304.24946871932
113396.62396276848
109002.60601750921
105575.96808242866
102813.18060938026
100528.86144782501
98602.20426977104
96950.64835735118
95515.65079971818
94254.44597834
93135.01241220415
92132.86584222502
91228.9438407439
90408.16923267354
89658.44991408098
88969.96726351578
88334.66011963476
87745.84415224417
87197.92676576563
86686.19056874907
86206.62682100643
85755.80582685905
85330.77499681884
84928.97787937301
84548.18926267036
84186.46271868283
83842.08787335652
83513.55534770206
83199.52779926301
82898.81585287493
82610.35797847137
82333.20357697534
82066.49869042309
81809.47387133012
81561.4338388056
81321.74862102464
81089.84594010869
80865.2046406421
80647.34899841933
80435.84377496381
80230.28990629896
80030.32073307855
79835.59869452628
79645.81242115775
79460.67417122694
79279.91756468039
79103.29557509974
78930.57874615099
78761.55360388203
78596.02124023033
78433.79604687663
78274.70458080636
78118.5

In [105]:
np.random.seed(18)
user1 = np.random.randint(0, len(userItemArray))
while True:
    movie1 = np.random.randint(0, len(userItemArray[user1]))
    if userItemArray[user1][movie1] == 0:
        break
print(f"user {user1} is predicted to have rated movie {movie1}: {nR[user1][movie1]:.1f}")
top10_user1 = np.argsort(nR[user1])[::-1][:10]

print(f"Top 10 predicted ratings for user {user1}:")

for i in top10_user1:
    print(f"movie {i}: {nR[user1][i]:.1f}")

user 298 is predicted to have rated movie 2885: 3.4
Top 10 predicted ratings for user 298:
movie 8831: 6.4
movie 2724: 6.3
movie 8766: 6.1
movie 4979: 6.1
movie 3727: 6.1
movie 7968: 6.0
movie 4448: 6.0
movie 6591: 6.0
movie 8639: 6.0
movie 6490: 6.0


In [106]:
np.random.seed(19)
user2 = np.random.randint(0, len(userItemArray))
while True:
    movie2 = np.random.randint(0, len(userItemArray[user2]))
    if userItemArray[user2][movie2] == 0:
        break
print(f"user {user2} is predicted to have rated movie {movie2}: {nR[user2][movie2]:.1f}")
top10_user2 = np.argsort(nR[user2])[::-1][:10]

print(f"Top 10 predicted ratings for user {user2}:")

for i in top10_user2:
    print(f"movie {i}: {nR[user2][i]:.1f}")

user 605 is predicted to have rated movie 757: 4.1
Top 10 predicted ratings for user 605:
movie 8831: 6.0
movie 5510: 5.9
movie 5759: 5.9
movie 5353: 5.8
movie 4296: 5.8
movie 7692: 5.7
movie 4979: 5.7
movie 7968: 5.7
movie 8766: 5.7
movie 7420: 5.6


In [107]:
np.random.seed(20)
user3 = np.random.randint(0, len(userItemArray))
while True:
    movie3 = np.random.randint(0, len(userItemArray[user3]))
    if userItemArray[user3][movie3] == 0:
        break
print(f"user {user3} is predicted to have rated movie {movie3}: {nR[user3][movie3]:.1f}")
top10_user3 = np.argsort(nR[user3])[::-1][:10]

print(f"Top 10 predicted ratings for user {user3}:")

for i in top10_user3:
    print(f"movie {i}: {nR[user3][i]:.1f}")

user 355 is predicted to have rated movie 4367: 3.8
Top 10 predicted ratings for user 355:
movie 6591: 5.8
movie 7568: 5.6
movie 274: 5.4
movie 7946: 5.4
movie 6064: 5.4
movie 1244: 5.3
movie 8744: 5.3
movie 5353: 5.3
movie 2991: 5.3
movie 4979: 5.3


evaluate 1

In [ ]:
answer = 0
for idx in removeIndices:
    partialAnswer = 0
    yTrue = userItemArray[idx]
    yTruePart = []
    yPred = nR[idx]
    yPredPart = []
    for i in range(len(userItemArray[idx])):
        if userItemArray[idx][i] > 0:
            yTruePart.append(userItemArray[idx][i])
            yPredPart.append(nR[idx][i])
    partialAnswer = np.mean((np.array(yTruePart) - np.array(yPredPart))**2)
    answer += partialAnswer
answer /= len(removeIndices)
print(f"Mean Squared Error: {answer}")

Mean Squared Error: 0.39222938943663144


evaluate 2

In [ ]:
userItemBoolean = (userItemArray >= 4.0).astype(bool)

In [ ]:
precisionAt5 = []
for i in range(len(userItemBoolean)):
    top5Indices = np.argsort(nR[i])[::-1][:5]
    summation = 0
    for j in top5Indices:
        if userItemBoolean[i][j]:
            summation += 1
    precision = summation / 5.0
    precisionAt5.append(precision)
    print(f"User {i+1}: Precision@5 = {precision}")

User 1: Precision@5 = 0.0
User 2: Precision@5 = 0.0
User 3: Precision@5 = 0.0
User 4: Precision@5 = 0.0
User 5: Precision@5 = 0.0
User 6: Precision@5 = 0.0
User 7: Precision@5 = 0.0
User 8: Precision@5 = 0.0
User 9: Precision@5 = 0.0
User 10: Precision@5 = 0.0
User 11: Precision@5 = 0.0
User 12: Precision@5 = 0.0
User 13: Precision@5 = 0.0
User 14: Precision@5 = 0.0
User 15: Precision@5 = 0.6
User 16: Precision@5 = 0.0
User 17: Precision@5 = 0.2
User 18: Precision@5 = 0.0
User 19: Precision@5 = 0.0
User 20: Precision@5 = 0.0
User 21: Precision@5 = 0.0
User 22: Precision@5 = 0.0
User 23: Precision@5 = 0.0
User 24: Precision@5 = 0.0
User 25: Precision@5 = 0.0
User 26: Precision@5 = 0.0
User 27: Precision@5 = 0.0
User 28: Precision@5 = 0.0
User 29: Precision@5 = 0.0
User 30: Precision@5 = 0.0
User 31: Precision@5 = 0.0
User 32: Precision@5 = 0.0
User 33: Precision@5 = 0.0
User 34: Precision@5 = 0.0
User 35: Precision@5 = 0.0
User 36: Precision@5 = 0.0
User 37: Precision@5 = 0.0
User 38: P

In [ ]:
precisionAt10 = []
for i in range(len(userItemBoolean)):
    top10Indices = np.argsort(nR[i])[::-1][:10]
    summation = 0
    for j in top10Indices:
        if userItemBoolean[i][j]:
            summation += 1
    precision = summation / 10.0
    precisionAt10.append(precision)
    print(f"User {i+1}: Precision@10 = {precision}")

User 1: Precision@10 = 0.0
User 2: Precision@10 = 0.0
User 3: Precision@10 = 0.0
User 4: Precision@10 = 0.0
User 5: Precision@10 = 0.0
User 6: Precision@10 = 0.0
User 7: Precision@10 = 0.0
User 8: Precision@10 = 0.0
User 9: Precision@10 = 0.0
User 10: Precision@10 = 0.0
User 11: Precision@10 = 0.0
User 12: Precision@10 = 0.0
User 13: Precision@10 = 0.0
User 14: Precision@10 = 0.0
User 15: Precision@10 = 0.6
User 16: Precision@10 = 0.0
User 17: Precision@10 = 0.1
User 18: Precision@10 = 0.0
User 19: Precision@10 = 0.0
User 20: Precision@10 = 0.0
User 21: Precision@10 = 0.0
User 22: Precision@10 = 0.0
User 23: Precision@10 = 0.0
User 24: Precision@10 = 0.0
User 25: Precision@10 = 0.0
User 26: Precision@10 = 0.0
User 27: Precision@10 = 0.0
User 28: Precision@10 = 0.0
User 29: Precision@10 = 0.0
User 30: Precision@10 = 0.0
User 31: Precision@10 = 0.0
User 32: Precision@10 = 0.0
User 33: Precision@10 = 0.0
User 34: Precision@10 = 0.0
User 35: Precision@10 = 0.0
User 36: Precision@10 = 0.0
U

In [ ]:
mrrScores = []
for i in range(len(userItemBoolean)):
    rank = 0.0
    for j in range(len(userItemBoolean[i])):
        if userItemBoolean[i][j]:
            rank = 1.0 / (j + 1)
            break
    mrrScores.append(rank)

meanMrr = np.mean(mrrScores)
print(f"Mean Reciprocal Rank (MRR): {meanMrr}")

Mean Reciprocal Rank (MRR): 0.29001479034948874


## References

https://medium.com/data-science/recommendation-system-matrix-factorization-d61978660b4b
https://www.geeksforgeeks.org/python/python-mean-squared-error/
https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k
https://www.geeksforgeeks.org/nlp/evaluation-metrics-for-retrieval-augmented-generation-rag-systems/